# المزامنة السريعة للحلقة — إصلاح الحوار الناقص

**هذا دفتر تشغيل فعلي لمرحلة إصلاح الصوت الموجود، وليس دفتر نقل الملفات.** يعتمد على ملف الدبلجة الحالي وتقارير التوقيت والماسترات الصوتية العشرة الموجودة في مجلد `دبلجة الحلقة الجديدة`.

ما ينفذه: يتحقق من الملفات وبصماتها، يعيد استخدام خريطة الحوار السابقة، يبحث عن توقيتات الحوارات الناقصة بدرجة ثقة محافظة، يضيف المقاطع التي يمكن ربطها بأمان، ويحافظ على مسار الصورة دون إعادة ترميزه، ثم يفحص الملف الناتج ويحفظه في `مخرجات المزامنة`.

**مهم:** لا يولّد أصواتًا جديدة ولا يبدّل أصوات الشخصيات. ولا يحرّك الشفاه داخل الصورة؛ فحص الصورة هنا لقطات مرئية فقط. إذا تعذر تحديد توقيت حوار بثقة، يتوقف بدل وضعه عشوائيًا.

شغّل **Runtime → Run all** بعد اختيار GPU من إعدادات الجلسة إن كان متاحًا. لا يحتاج هذا المسار إلى GPU إلزاميًا؛ فهو يستخدم FFmpeg لنسخ الصورة وخلط الصوت. سيطلب Colab موافقة ربط Drive المعتادة.

In [ ]:
# 1) تثبيت الأدوات وفحص مساحة العمل
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install pillow requests
import os, sys, json, shutil, hashlib, zipfile, subprocess, time
from pathlib import Path
print(subprocess.run(['ffmpeg','-version'],capture_output=True,text=True,check=True).stdout.splitlines()[0])
free_gib=shutil.disk_usage('/content').free/1024**3
print(f'المساحة الحرة: {free_gib:.1f} GiB')
if free_gib < 8: raise RuntimeError('المساحة الحرة أقل من 8 GiB؛ أفرغ مساحة الجلسة قبل بدء نسخ الفيديوهات.')


In [ ]:
# 2) ربط Drive وتحديد ملفات الحلقة
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive')
EPISODE_DIR=DRIVE_ROOT/'دبلجة الحلقة الجديدة'
OUTPUT_DIR=EPISODE_DIR/'مخرجات المزامنة'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
if not EPISODE_DIR.is_dir(): raise FileNotFoundError(f'مجلد الحلقة غير موجود: {EPISODE_DIR}')
SOURCE_NAME='🔴 FULL - 女孩相亲认错人，将错就错与千亿霸总当天领证！ - Original Quality.mkv'
DUB_NAME='الحلقة_مدبلجة_بالعربية.mkv'
REPORTS_NAME='تقارير_التحقق_والموائمة.zip'
VOICES_ARCHIVE_NAME='الأصوات_العربية_المتبقية_06-10.zip'
VOICE_NAMES=['01_لي_يانغ_Leda.wav','02_لو_مينغ_Achird.wav','03_تشوانغ_شيان_يوي_Kore.wav','04_فو_شاوتشينغ_Puck.wav','05_شيا_مو_يان_Zephyr.wav']
EXPECTED=[SOURCE_NAME,DUB_NAME,REPORTS_NAME,VOICES_ARCHIVE_NAME]+VOICE_NAMES
missing=[n for n in EXPECTED if not (EPISODE_DIR/n).is_file()]
if missing: raise FileNotFoundError('ملفات ناقصة في مجلد الحلقة:\n'+'\n'.join(missing))
for n in EXPECTED: print('✓',n, round((EPISODE_DIR/n).stat().st_size/1024**2,1),'MiB')
print('مجلد النتائج:',OUTPUT_DIR)


In [ ]:
# 3) فحص ZIP والتأكد من توفر ملفات تقارير التوقيت والماسترات العشرة
REPORTS=EPISODE_DIR/REPORTS_NAME
VOICE_ARCHIVE=EPISODE_DIR/VOICES_ARCHIVE_NAME
for p in [REPORTS,VOICE_ARCHIVE]:
    with zipfile.ZipFile(p) as z:
        bad=z.testzip()
        if bad: raise RuntimeError(f'ملف ZIP تالف: {p.name} / {bad}')
        print(p.name, 'سليم؛ عدد الملفات:',len(z.namelist()))
        if p==REPORTS:
            required={'render_manifest.json','final_probe.json','source_to_arabic_lines.json','voice_line_timing.json'}
            names={Path(n).name for n in z.namelist()}
            absent=required-names
            if absent: raise RuntimeError('تقارير أساسية غير موجودة داخل الأرشيف: '+', '.join(sorted(absent)))
with zipfile.ZipFile(VOICE_ARCHIVE) as z:
    wavs=[Path(n).name for n in z.namelist() if n.lower().endswith('.wav')]
    print('ملفات WAV في الأرشيف:',len(wavs))
if len(wavs)<5: raise RuntimeError('أرشيف الأصوات لا يحتوي عددًا كافيًا من ملفات WAV المتبقية 06–10.')


In [ ]:
# 4) تنزيل سكربت الإصلاح من المستودع وتجهيز نسخة محلية للعمل
REPO=Path('/content/AI-Creative-Studio')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/h776117073-eng/AI-Creative-Studio.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
RUNNER_PATH=REPO/'automation/episode_repair_20261031/runner.py'
if not RUNNER_PATH.is_file(): raise FileNotFoundError(RUNNER_PATH)
print('تم تجهيز سكربت الإصلاح:',RUNNER_PATH)


In [ ]:
# 5) بناء بيان ملفات محلي؛ لا روابط عامة ولا مفاتيح OAuth داخل الدفتر
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''): h.update(block)
    return h.hexdigest()
LOCAL_FILES={
 'source':EPISODE_DIR/SOURCE_NAME,
 'current_dub':EPISODE_DIR/DUB_NAME,
 'reports':REPORTS,
 'voice_archive':VOICE_ARCHIVE,
}
individual={n:{'name':n,'url':str(EPISODE_DIR/n),'sha256':sha256_file(EPISODE_DIR/n)} for n in VOICE_NAMES}
transfer={'transfer_policy':'private_signed_download_only','files':{
 'source':{'name':SOURCE_NAME,'url':str(LOCAL_FILES['source']),'sha256':sha256_file(LOCAL_FILES['source'])},
 'current_dub':{'name':DUB_NAME,'url':str(LOCAL_FILES['current_dub']),'sha256':sha256_file(LOCAL_FILES['current_dub'])},
 'reports':{'name':REPORTS_NAME,'url':str(LOCAL_FILES['reports']),'sha256':sha256_file(LOCAL_FILES['reports'])},
 'voice_archive':{'name':VOICES_ARCHIVE_NAME,'url':str(LOCAL_FILES['voice_archive']),'sha256':sha256_file(LOCAL_FILES['voice_archive'])},
 'individual_voices':individual}}
RUN_ROOT=Path('/content/episode_repair_20261031')
RUN_ROOT.mkdir(parents=True,exist_ok=True)
(RUN_ROOT/'transfer_manifest.json').write_text(json.dumps(transfer,ensure_ascii=False,indent=2),encoding='utf-8')
print('بيان النقل المحلي جاهز؛ لن تُرفع الملفات إلى GitHub.')


In [ ]:
# 6) تشغيل الإصلاح مع إعادة استخدام الملفات المحلية؛ لا يشترط T4
import importlib.util, traceback
spec=importlib.util.spec_from_file_location('episode_repair_runner',RUNNER_PATH)
runner=importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
def local_download(path_text,dest,expected_sha256=None):
    src=Path(path_text)
    if not src.is_file(): raise FileNotFoundError(f'ملف محلي غير موجود: {src}')
    dest=Path(dest); dest.parent.mkdir(parents=True,exist_ok=True)
    if not dest.exists() or dest.stat().st_size!=src.stat().st_size: shutil.copy2(src,dest)
    if expected_sha256 and sha256_file(dest)!=expected_sha256: raise RuntimeError('فشل تحقق SHA256: '+dest.name)
    print('تم التحقق:',dest.name,round(dest.stat().st_size/1024**2,1),'MiB',flush=True)
    return dest
def fast_tool_check():
    for tool in ['ffmpeg','ffprobe']:
        if shutil.which(tool) is None: raise RuntimeError('الأداة غير موجودة: '+tool)
    print('FFmpeg/FFprobe جاهزان. الفيديو سيُنسخ دون إعادة ترميز الصورة.',flush=True)
runner.download_signed_url=local_download
runner.ensure_tools=fast_tool_check
# منع إعادة استخدام ناتج قديم إذا توقفت محاولة سابقة قبل اكتمالها
for stale in [runner.FINAL_PATH, runner.QA_PATH, runner.PLAN_PATH, runner.AUDIT_PATH, runner.ROOT/'contact_sheet.jpg']:
    Path(stale).unlink(missing_ok=True)
runner.sync_json=lambda *args,**kwargs: None
print('بدأ الإصلاح. قد يتوقف بأمان إذا تعذر إسناد أي حوار إلى توقيت موثوق.',flush=True)
try:
    runner.main()
except Exception:
    print(traceback.format_exc()[-12000:])
    raise


In [ ]:
# 7) حفظ النتائج والتقارير على Drive والتحقق من الملف النهائي
import json, shutil, hashlib
artifacts={
 'الحلقة_مدبلجة_بالعربية_بعد_إصلاح_الحوارات.mkv':RUN_ROOT/'الحلقة_مدبلجة_بالعربية_النهائية.mkv',
 'تقرير_فحص_المزامنة.json':RUN_ROOT/'qa_report.json',
 'خطة_توقيت_الحوارات.json':RUN_ROOT/'timeline_plan.json',
 'تدقيق_الملفات.json':RUN_ROOT/'asset_audit.json',
 'سجل_التشغيل.txt':RUN_ROOT/'repair.log',
 'لوحة_عينات_بصرية.jpg':RUN_ROOT/'contact_sheet.jpg',
}
saved=[]
for out_name,src in artifacts.items():
    if src.is_file():
        dst=OUTPUT_DIR/out_name
        shutil.copy2(src,dst)
        saved.append((out_name,dst.stat().st_size))
        print('حُفظ:',dst)
final=artifacts['الحلقة_مدبلجة_بالعربية_بعد_إصلاح_الحوارات.mkv']
if final.is_file():
    probe=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration,size','-of','json',str(final)],capture_output=True,text=True,check=True)
    info=json.loads(probe.stdout)['format']
    print('المدة بالثواني:',round(float(info['duration']),2),'الحجم:',info['size'])
    print('SHA256:',sha256_file(final))
    print('الملف النهائي محفوظ على Drive:',OUTPUT_DIR/'الحلقة_مدبلجة_بالعربية_بعد_إصلاح_الحوارات.mkv')
else:
    print('لم يُنشأ ملف نهائي. راجع تقرير الفحص وخطة التوقيت وسجل التشغيل؛ قد يكون الإصلاح توقف بسبب توقيت غير موثوق.')
print('مجلد المخرجات:',OUTPUT_DIR)
